In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import h5py
import os
import matplotlib.pyplot as plt
from tqdm import tqdm
from math import log10
import pandas as pd
import warnings
warnings.filterwarnings("ignore")



# Install LPIPS if not available
try:
    import lpips
except ImportError:
    !pip install lpips -q
    import lpips

In [16]:
from google.colab import drive
drive.mount('/content/drive')

# Copy data to local disk
!cp -r /content/drive/MyDrive/procgen_tokenized /content/procgen_tokenized
!cp -r /content/drive/MyDrive/procgen_data /content/procgen_data
print("Data copied!")

TOKEN_DIR = "/content/procgen_tokenized"
DATA_DIR = "/content/procgen_data"
VQVAE_PATH = "/content/drive/MyDrive/procgen_checkpoints/vqvae_epoch30.pt"
RESULTS_DIR = "/content/drive/MyDrive/evaluation_results_all"
os.makedirs(RESULTS_DIR, exist_ok=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data copied!


In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")


Using device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


## Model Definitions

In [18]:
# ═══════════════════════════════════════════════════════════
# VQ-VAE
# ═══════════════════════════════════════════════════════════

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.BatchNorm2d(channels), nn.ReLU(),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(channels), nn.ReLU(),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
        )
    def forward(self, x):
        return x + self.block(x)

class Encoder(nn.Module):
    def __init__(self, in_channels=3, hidden_dim=256, embed_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, hidden_dim, 4, 2, 1), nn.ReLU(), ResidualBlock(hidden_dim),
            nn.Conv2d(hidden_dim, hidden_dim, 4, 2, 1), nn.ReLU(), ResidualBlock(hidden_dim),
            nn.Conv2d(hidden_dim, hidden_dim, 4, 2, 1), nn.ReLU(), ResidualBlock(hidden_dim),
            nn.Conv2d(hidden_dim, embed_dim, 3, 1, 1), ResidualBlock(embed_dim),
        )
    def forward(self, x): return self.net(x)

class VectorQuantizer(nn.Module):
    def __init__(self, num_embeddings=1024, embed_dim=512, ema_decay=0.99):
        super().__init__()
        self.num_embeddings, self.embed_dim = num_embeddings, embed_dim
        self.codebook = nn.Embedding(num_embeddings, embed_dim)
        self.codebook.weight.data.uniform_(-1.0/num_embeddings, 1.0/num_embeddings)
        self.register_buffer("ema_cluster_size", torch.zeros(num_embeddings))
        self.register_buffer("ema_embed_sum", self.codebook.weight.data.clone())
    def forward(self, z):
        z = z.permute(0,2,3,1).contiguous()
        flat_z = z.view(-1, self.embed_dim)
        distances = flat_z.pow(2).sum(1,keepdim=True) + self.codebook.weight.pow(2).sum(1) - 2*flat_z@self.codebook.weight.t()
        token_ids = distances.argmin(dim=1)
        quantized = self.codebook(token_ids).view(z.shape)
        vq_loss = 0.25 * F.mse_loss(z, quantized.detach())
        quantized_st = z + (quantized - z).detach()
        return quantized_st.permute(0,3,1,2), vq_loss, token_ids.view(z.shape[0], z.shape[1], z.shape[2])

class Decoder(nn.Module):
    def __init__(self, out_channels=3, hidden_dim=256, embed_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(embed_dim, hidden_dim, 3, 1, 1), ResidualBlock(hidden_dim),
            nn.ConvTranspose2d(hidden_dim, hidden_dim, 4, 2, 1), nn.ReLU(), ResidualBlock(hidden_dim),
            nn.ConvTranspose2d(hidden_dim, hidden_dim, 4, 2, 1), nn.ReLU(), ResidualBlock(hidden_dim),
            nn.ConvTranspose2d(hidden_dim, out_channels, 4, 2, 1),
        )
    def forward(self, x): return self.net(x)

class VQVAE(nn.Module):
    def __init__(self, in_channels=3, hidden_dim=256, embed_dim=512, num_embeddings=1024):
        super().__init__()
        self.encoder = Encoder(in_channels, hidden_dim, embed_dim)
        self.quantizer = VectorQuantizer(num_embeddings, embed_dim)
        self.decoder = Decoder(in_channels, hidden_dim, embed_dim)
    def decode_from_tokens(self, token_ids):
        q = self.quantizer.codebook(token_ids).permute(0,3,1,2)
        return self.decoder(q)
    def forward(self, x):
        z = self.encoder(x)
        quantized, vq_loss, token_ids = self.quantizer(z)
        return self.decoder(quantized), vq_loss, token_ids

# ═══════════════════════════════════════════════════════════
# Transformer
# ═══════════════════════════════════════════════════════════

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_temporal=20, spatial_positions=65):
        super().__init__()
        self.temporal_embed = nn.Embedding(max_temporal, d_model)
        self.spatial_embed = nn.Embedding(spatial_positions, d_model)
    def forward(self, temporal_ids, spatial_ids):
        return self.temporal_embed(temporal_ids) + self.spatial_embed(spatial_ids)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attention = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout),
        )
    def forward(self, x, causal_mask=None):
        attn_out, _ = self.attention(x, x, x, attn_mask=causal_mask)
        x = self.norm1(x + attn_out)
        x = self.norm2(x + self.ffn(x))
        return x

class WorldModelTransformer(nn.Module):
    def __init__(self, vocab_size=512, num_actions=15, d_model=256, n_heads=8,
                 n_layers=6, d_ff=1024, context_frames=4, tokens_per_frame=64, dropout=0.1):
        super().__init__()
        self.vocab_size, self.num_actions, self.d_model = vocab_size, num_actions, d_model
        self.context_frames, self.tokens_per_frame = context_frames, tokens_per_frame
        self.total_vocab = vocab_size + num_actions
        self.token_embed = nn.Embedding(self.total_vocab, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_temporal=context_frames+1, spatial_positions=tokens_per_frame+1)
        self.transformer_blocks = nn.ModuleList([TransformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.output_head = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self._causal_mask = None

    def _get_causal_mask(self, seq_len, device):
        if self._causal_mask is None or self._causal_mask.shape[0] < seq_len:
            self._causal_mask = torch.triu(torch.ones(seq_len, seq_len, device=device)*float('-inf'), diagonal=1)
        return self._causal_mask[:seq_len, :seq_len]

    def _build_sequence(self, tokens, actions):
        B, T, K = tokens.shape
        sequences, temporal, spatial = [], [], []
        for t in range(T):
            sequences.append(tokens[:, t, :])
            temporal.append(torch.full((B, K), t, device=tokens.device))
            spatial.append(torch.arange(K, device=tokens.device).unsqueeze(0).expand(B, -1))
            if t < T - 1:
                sequences.append(actions[:, t].unsqueeze(1) + self.vocab_size)
                temporal.append(torch.full((B, 1), t, device=tokens.device))
                spatial.append(torch.full((B, 1), K, device=tokens.device))
        return torch.cat(sequences, 1), torch.cat(temporal, 1), torch.cat(spatial, 1)

    def forward(self, tokens, actions):
        input_ids, temporal_ids, spatial_ids = self._build_sequence(tokens, actions)
        x = self.token_embed(input_ids) + self.pos_encoding(temporal_ids, spatial_ids)
        x = self.dropout(x)
        mask = self._get_causal_mask(x.shape[1], x.device)
        for block in self.transformer_blocks:
            x = block(x, mask)
        return self.output_head(x)

    @torch.no_grad()
    def predict_next_frame(self, context_tokens, context_actions, next_action=None):
        """
        Args:
            context_tokens: [B, T, 64] — past frame tokens
            context_actions: [B, T-1] — actions between context frames
            next_action: [B] — action leading to the predicted frame.
                        If None, repeats last context action (fallback).
        """
        self.eval()
        B = context_tokens.shape[0]
        T = context_tokens.shape[1]

        # Build full action sequence: T-1 between context + 1 to predicted frame
        if next_action is not None:
            transition = next_action.unsqueeze(1)  # [B, 1]
        else:
            transition = context_actions[:, -1:]  # fallback: repeat last

        full_actions = torch.cat([context_actions, transition], dim=1)  # [B, T]

        dummy_frame = torch.zeros(B, 1, self.tokens_per_frame,
                                  dtype=torch.long, device=context_tokens.device)
        full_tokens = torch.cat([context_tokens, dummy_frame], dim=1)

        pred_frame_start = T * self.tokens_per_frame + T

        generated = []
        for i in range(self.tokens_per_frame):
            logits = self.forward(full_tokens, full_actions)
            logit_pos = pred_frame_start + i - 1
            next_token = logits[:, logit_pos, :].argmax(dim=-1)
            generated.append(next_token)
            full_tokens[:, T, i] = next_token

        return torch.stack(generated, dim=1)

print("All models defined.")


All models defined.


### Load Models

In [19]:
# Load VQ-VAE
vqvae = VQVAE(in_channels=3, hidden_dim=256, embed_dim=64, num_embeddings=512).to(device)
vqvae_ckpt = torch.load(VQVAE_PATH, map_location=device)
vqvae.load_state_dict(vqvae_ckpt["model_state_dict"])
vqvae.eval()
print(f"VQ-VAE loaded from {VQVAE_PATH}")


VQ-VAE loaded from /content/drive/MyDrive/procgen_checkpoints/vqvae_epoch30.pt


In [20]:
# ═══ Model Configs — UPDATE checkpoint paths if needed ═══

MODEL_CONFIGS = {
    "V1": {
        "label": "V1 (d384, L8, drop0.1)",
        "params": dict(vocab_size=512, num_actions=15, d_model=384, n_heads=8,
                       n_layers=8, d_ff=1536, context_frames=4, dropout=0.1),
        "ckpt": "/content/drive/MyDrive/transformer_checkpoints/transformer_final.pt",
        "color": "#e74c3c",
    },
    "V2": {
        "label": "V2 (d256, L4, drop0.3)",
        "params": dict(vocab_size=512, num_actions=15, d_model=256, n_heads=8,
                       n_layers=4, d_ff=1536, context_frames=4, dropout=0.3),
        "ckpt": "/content/drive/MyDrive/transformer_checkpoints_v2/transformer_epoch40.pt",
        "color": "#3498db",
    },
    "V3": {
        "label": "V3 (d256, L6, drop0.2)",
        "params": dict(vocab_size=512, num_actions=15, d_model=256, n_heads=8,
                       n_layers=6, d_ff=1536, context_frames=4, dropout=0.2),
        "ckpt": "/content/drive/MyDrive/transformer_checkpoints_v3/transformer_best.pt",
        "color": "#2ecc71",
    },
    "V4": {
        "label": "V4 (d384, L8, ctx5, drop0.1)",
        "params": dict(vocab_size=512, num_actions=15, d_model=384, n_heads=8,
                       n_layers=8, d_ff=1536, context_frames=5, dropout=0.1),
        "ckpt": "/content/drive/MyDrive/transformer_checkpoints_v4/transformer_best23.pt",
        "color": "#9b59b6",
    },
}

# Load all models
models = {}
for key, cfg in MODEL_CONFIGS.items():
    print(f"Loading {cfg['label']}...")
    m = WorldModelTransformer(**cfg["params"]).to(device)
    ckpt = torch.load(cfg["ckpt"], map_location=device)
    m.load_state_dict(ckpt["model_state_dict"])
    m.eval()
    n_params = sum(p.numel() for p in m.parameters())
    print(f"  Params: {n_params:,} | Epoch: {ckpt.get('epoch', '?')}")
    models[key] = m

print(f"\nAll {len(models)} models loaded!")


Loading V1 (d384, L8, drop0.1)...
  Params: 14,622,080 | Epoch: 60
Loading V2 (d256, L4, drop0.3)...
  Params: 4,494,080 | Epoch: 40
Loading V3 (d256, L6, drop0.2)...
  Params: 6,598,912 | Epoch: 60
Loading V4 (d384, L8, ctx5, drop0.1)...
  Params: 14,622,464 | Epoch: 30

All 4 models loaded!


### Evaluation Dataset

In [21]:
class EvalDataset:
    """Loads raw frames (for pixel metrics) + tokenized data (for transformer input)."""
    def __init__(self, raw_h5_path, token_h5_path, context_frames=4):
        self.context_frames = context_frames
        with h5py.File(raw_h5_path, "r") as f:
            self.raw_frames = f["frames"][:].astype(np.float32) / 255.0
            self.actions = f["actions"][:]
            self.dones = f["dones"][:]
        with h5py.File(token_h5_path, "r") as f:
            self.tokens = f["tokens"][:].reshape(-1, 64)
        self.valid_starts = self._find_valid_starts()
        print(f"  {os.path.basename(raw_h5_path)}: {len(self.raw_frames)} frames, "
              f"{len(self.valid_starts)} valid sequences")

    def _find_valid_starts(self):
        max_horizon = 20
        min_seq_len = self.context_frames + max_horizon
        valid = []
        for i in range(len(self.tokens) - min_seq_len):
            if not self.dones[i:i + min_seq_len].any():
                valid.append(i)
        return np.array(valid)

    def get_sequence(self, idx, horizon=1):
        start = self.valid_starts[idx]
        ctx = self.context_frames
        context_tokens = torch.tensor(self.tokens[start:start+ctx], dtype=torch.long)
        context_actions = torch.tensor(self.actions[start:start+ctx], dtype=torch.long)
        future_frames = self.raw_frames[start+ctx:start+ctx+horizon]
        future_tokens = self.tokens[start+ctx:start+ctx+horizon]
        future_actions = self.actions[start+ctx:start+ctx+horizon-1]
        return context_tokens, context_actions, future_frames, future_tokens, future_actions


# We need separate datasets for models with different context_frames.
# V1-V3 use ctx=4, V4 uses ctx=5.
print("Loading evaluation datasets...")
datasets_ctx4 = {}
datasets_ctx5 = {}

for game in ["coinrun", "starpilot"]:
    for split in ["train", "test"]:
        name = f"{game}_{split}"
        raw_path = os.path.join(DATA_DIR, f"{game}_{split}.h5")
        tok_path = os.path.join(TOKEN_DIR, f"{game}_{split}_tokens.h5")
        if os.path.exists(raw_path) and os.path.exists(tok_path):
            datasets_ctx4[name] = EvalDataset(raw_path, tok_path, context_frames=4)
            datasets_ctx5[name] = EvalDataset(raw_path, tok_path, context_frames=5)

print(f"\nLoaded datasets for ctx=4 and ctx=5.")


Loading evaluation datasets...
  coinrun_train.h5: 100000 frames, 96064 valid sequences
  coinrun_train.h5: 100000 frames, 95900 valid sequences
  coinrun_test.h5: 20000 frames, 19256 valid sequences
  coinrun_test.h5: 20000 frames, 19225 valid sequences
  starpilot_train.h5: 100000 frames, 70382 valid sequences
  starpilot_train.h5: 100000 frames, 69166 valid sequences
  starpilot_test.h5: 20000 frames, 13699 valid sequences
  starpilot_test.h5: 20000 frames, 13439 valid sequences

Loaded datasets for ctx=4 and ctx=5.


### Prediction Functions

In [22]:
@torch.no_grad()
def predict_multi_step(transformer, vqvae, context_tokens, context_actions,
                       future_actions, horizon):
    ctx_tokens = context_tokens.unsqueeze(0).to(device)

    # Build the full ordered list of transition actions:
    # context_actions[:-1] = between context frames
    # context_actions[-1]  = from last context to first predicted frame
    # future_actions[0:]   = between predicted frames
    between_ctx = context_actions[:-1]
    to_first_pred = context_actions[-1:]
    if len(future_actions) > 0:
        all_actions = torch.cat([between_ctx, to_first_pred, future_actions])
    else:
        all_actions = torch.cat([between_ctx, to_first_pred])

    predicted_tokens_list = []
    predicted_frames_list = []

    for step in range(horizon):
        if ctx_tokens.shape[1] > transformer.context_frames:
            ctx_tokens = ctx_tokens[:, -transformer.context_frames:]

        n_frames = ctx_tokens.shape[1]

        # Actions between the context frames for this window
        ctx_act = all_actions[step : step + n_frames - 1].unsqueeze(0).to(device)

        # The real action leading to the next predicted frame
        next_act_idx = step + n_frames - 1
        if next_act_idx < len(all_actions):
            next_act = all_actions[next_act_idx].unsqueeze(0).to(device)
        else:
            next_act = None  # fallback for free dreaming

        pred_tokens = transformer.predict_next_frame(ctx_tokens, ctx_act, next_act)
        predicted_tokens_list.append(pred_tokens.cpu())

        pred_grid = pred_tokens.view(1, 8, 8)
        pred_frame = vqvae.decode_from_tokens(pred_grid).clamp(0, 1).cpu()
        predicted_frames_list.append(pred_frame[0].permute(1, 2, 0).detach().numpy())

        ctx_tokens = torch.cat([ctx_tokens, pred_tokens.unsqueeze(1).to(device)], dim=1)

    return np.stack(predicted_frames_list), torch.cat(predicted_tokens_list).numpy()


### Metrics

In [23]:
lpips_model = lpips.LPIPS(net='alex').to(device)
lpips_model.eval()

def compute_psnr(original, predicted):
    mse = np.mean((original - predicted) ** 2)
    if mse == 0: return float('inf')
    return 10 * log10(1.0 / mse)

def compute_ssim_simple(original, predicted):
    C1, C2 = 0.01**2, 0.03**2
    if original.ndim == 3:
        original = np.mean(original, axis=2)
        predicted = np.mean(predicted, axis=2)
    mu_x, mu_y = np.mean(original), np.mean(predicted)
    sigma_x, sigma_y = np.std(original), np.std(predicted)
    sigma_xy = np.mean((original - mu_x) * (predicted - mu_y))
    return ((2*mu_x*mu_y+C1)*(2*sigma_xy+C2)) / ((mu_x**2+mu_y**2+C1)*(sigma_x**2+sigma_y**2+C2))

def compute_token_accuracy(predicted_tokens, ground_truth_tokens):
    return (predicted_tokens == ground_truth_tokens).mean()

def compute_lpips(original, predicted):
    """LPIPS between two frames [H, W, 3] in [0, 1]. Lower is better."""
    # Convert to [1, 3, H, W] tensors in [-1, 1]
    orig_t = torch.tensor(original).permute(2, 0, 1).unsqueeze(0).float() * 2 - 1
    pred_t = torch.tensor(predicted).permute(2, 0, 1).unsqueeze(0).float() * 2 - 1
    with torch.no_grad():
        score = lpips_model(orig_t.to(device), pred_t.to(device))
    return score.item()

print("Metrics defined.")


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
Metrics defined.


### Run Evaluation — All Models

In [ ]:
HORIZONS = [1, 5, 10, 20]
N_SAMPLES = 200

all_results = {}

for model_key, model in models.items():
    cfg = MODEL_CONFIGS[model_key]
    ctx = cfg["params"]["context_frames"]
    datasets = datasets_ctx5 if ctx == 5 else datasets_ctx4

    print(f"\n{'='*60}")
    print(f"Evaluating {cfg['label']} (context_frames={ctx})")
    print(f"{'='*60}")

    all_results[model_key] = {}

    for ds_name, dataset in datasets.items():
        print(f"\n  Dataset: {ds_name}")
        all_results[model_key][ds_name] = {}

        for horizon in HORIZONS:
            psnr_vals, ssim_vals, acc_vals, lpips_vals = [], [], [], []
            n_eval = min(N_SAMPLES, len(dataset.valid_starts))
            indices = np.linspace(0, len(dataset.valid_starts)-1, n_eval, dtype=int)

            for idx in tqdm(indices, desc=f"    h={horizon}", leave=False):
                ctx_tokens, ctx_actions, gt_frames, gt_tokens, future_actions = \
                    dataset.get_sequence(idx, horizon=horizon)

                pred_frames, pred_tokens = predict_multi_step(
                    model, vqvae, ctx_tokens, ctx_actions,
                    torch.tensor(future_actions, dtype=torch.long), horizon
                )

                # Metrics on the LAST predicted frame (quality at full horizon)
                psnr_vals.append(compute_psnr(gt_frames[-1], pred_frames[-1]))
                ssim_vals.append(compute_ssim_simple(gt_frames[-1], pred_frames[-1]))
                acc_vals.append(compute_token_accuracy(pred_tokens[-1], gt_tokens[-1]))
                lpips_vals.append(compute_lpips(gt_frames[-1], pred_frames[-1]))

            all_results[model_key][ds_name][horizon] = {
                "psnr": np.mean(psnr_vals),
                "ssim": np.mean(ssim_vals),
                "token_acc": np.mean(acc_vals),
                "lpips": np.mean(lpips_vals),
            }
            r = all_results[model_key][ds_name][horizon]
            print(f"    h={horizon:2d}: PSNR={r['psnr']:.2f} | SSIM={r['ssim']:.4f} | LPIPS={r['lpips']:.4f} |Acc={r['token_acc']:.3f}")

print("\nEvaluation complete!")



Evaluating V1 (d384, L8, drop0.1) (context_frames=4)

  Dataset: coinrun_train


    h= 1: PSNR=24.02 | SSIM=0.9670 | LPIPS=0.0809 |Acc=0.869


    h=5:  16%|█▌        | 32/200 [00:17<01:33,  1.79it/s]

### Results Tables

In [ ]:
# ═══ Print results for all models ═══

for model_key in models:
    cfg = MODEL_CONFIGS[model_key]
    print(f"\n{'='*70}")
    print(f"  {cfg['label']}")
    print(f"{'='*70}")
    print(f"  {'Game':<12} {'Split':<8} {'Horizon':<10} {'PSNR':<10} {'SSIM':<10} {'Acc':<10}")
    print(f"  {'─'*60}")

    for ds_name in sorted(all_results[model_key]):
        game = ds_name.split("_")[0]
        split = ds_name.split("_")[1]
        for h in HORIZONS:
            r = all_results[model_key][ds_name][h]
            label = "seen" if split == "train" else "unseen"
            print(f"  {game:<12} {label:<8} {h:<10} {r['psnr']:<10.2f} {r['ssim']:<10.4f} {r['token_acc']:<10.3f}")

# ═══ Generalization gap ═══
print(f"\n\n{'='*70}")
print("  GENERALIZATION GAP (train PSNR - test PSNR)")
print(f"{'='*70}")
print(f"  {'Model':<30} {'Game':<12} {'Horizon':<10} {'ΔPSNR':<10}")
print(f"  {'─'*62}")

for model_key in models:
    cfg = MODEL_CONFIGS[model_key]
    for game in ["coinrun", "starpilot"]:
        for h in HORIZONS:
            train_r = all_results[model_key].get(f"{game}_train", {}).get(h, {})
            test_r = all_results[model_key].get(f"{game}_test", {}).get(h, {})
            if train_r and test_r:
                gap = train_r["psnr"] - test_r["psnr"]
                print(f"  {cfg['label']:<30} {game:<12} {h:<10} {gap:+.2f}")


### Plots — All Models Compared

In [ ]:
# ═══ PSNR vs Horizon — all models on test sets ═══

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
games = ["coinrun", "starpilot"]

for col, game in enumerate(games):
    ax = axes[col]
    for model_key in models:
        cfg = MODEL_CONFIGS[model_key]
        ds_name = f"{game}_test"
        if ds_name in all_results[model_key]:
            psnrs = [all_results[model_key][ds_name][h]["psnr"] for h in HORIZONS]
            ax.plot(HORIZONS, psnrs, marker="o", label=cfg["label"],
                    color=cfg["color"], linewidth=2)
    ax.set_xlabel("Prediction Horizon")
    ax.set_ylabel("PSNR (dB)")
    ax.set_title(f"{game.upper()} — Unseen Levels")
    ax.set_xticks(HORIZONS)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle("Prediction Quality vs Horizon (Test Sets)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "psnr_vs_horizon_all_models.png"), dpi=150)
plt.show()


In [ ]:
# ═══ Train vs Test — per model ═══

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
games = ["coinrun", "starpilot"]

for col, model_key in enumerate(models):
    cfg = MODEL_CONFIGS[model_key]
    for row, game in enumerate(games):
        ax = axes[row, col]
        for split, ls in [("train", "-"), ("test", "--")]:
            ds_name = f"{game}_{split}"
            if ds_name in all_results[model_key]:
                psnrs = [all_results[model_key][ds_name][h]["psnr"] for h in HORIZONS]
                ax.plot(HORIZONS, psnrs, marker="o", linestyle=ls,
                        label=split.capitalize(), linewidth=2)
        ax.set_xlabel("Horizon")
        ax.set_ylabel("PSNR (dB)")
        ax.set_title(f"{cfg['label']}\n{game.upper()}", fontsize=9)
        ax.set_xticks(HORIZONS)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

plt.suptitle("Train vs Test PSNR — Generalization Gap", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "generalization_gap.png"), dpi=150, bbox_inches="tight")
plt.show()


### Visualize Predictions

In [ ]:
def visualize_all_models(ds_name, sample_idx=0, horizon=10):
    """Show GT row + one prediction row per model."""

    # Use max context so all models predict the same frames
    max_ctx = max(cfg["params"]["context_frames"] for cfg in MODEL_CONFIGS.values())

    # Get ground truth from the max-context dataset
    ds = datasets_ctx5 if max_ctx == 5 else datasets_ctx4
    ctx_tokens, ctx_actions, gt_frames, gt_tokens, future_actions = \
        ds[ds_name].get_sequence(sample_idx, horizon=horizon)

    n_models = len(models)
    fig, axes = plt.subplots(n_models + 1, horizon, figsize=(2*horizon, 2.5*(n_models+1)))

    # Row 0: Ground truth
    for t in range(horizon):
        axes[0, t].imshow(gt_frames[t])
        axes[0, t].axis("off")
        axes[0, t].set_title(f"t+{t+1}", fontsize=8)
    axes[0, 0].set_ylabel("Ground Truth", fontsize=10, rotation=0, labelpad=80, va="center")

    # Rows 1+: Each model
    for m_idx, (model_key, model) in enumerate(models.items()):
        cfg = MODEL_CONFIGS[model_key]
        ctx = cfg["params"]["context_frames"]
        d = datasets_ctx5[ds_name] if ctx == 5 else datasets_ctx4[ds_name]

        c_tokens, c_actions, _, _, f_actions = d.get_sequence(sample_idx, horizon=horizon)
        pred_frames, _ = predict_multi_step(
            model, vqvae, c_tokens, c_actions,
            torch.tensor(f_actions, dtype=torch.long), horizon
        )

        for t in range(horizon):
            axes[m_idx+1, t].imshow(pred_frames[t])
            axes[m_idx+1, t].axis("off")
        axes[m_idx+1, 0].set_ylabel(cfg["label"], fontsize=9, rotation=0,
                                      labelpad=80, va="center")

    plt.suptitle(f"Predictions — {ds_name} (sample {sample_idx})", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f"visual_{ds_name}_s{sample_idx}_h{horizon}.png"),
                dpi=150, bbox_inches="tight")
    plt.show()


for ds_name in ["coinrun_test", "starpilot_test"]:
    for s_idx in [0, 5]:
        visualize_all_models(ds_name, sample_idx=s_idx, horizon=10)


In [ ]:
def visualize_with_context(ds_name, model_key="V1", sample_idx=0, horizon=10):
    """Show context → GT vs context → predicted for one model."""
    cfg = MODEL_CONFIGS[model_key]
    ctx = cfg["params"]["context_frames"]
    d = datasets_ctx5[ds_name] if ctx == 5 else datasets_ctx4[ds_name]

    c_tokens, c_actions, gt_frames, _, f_actions = d.get_sequence(sample_idx, horizon=horizon)
    pred_frames, _ = predict_multi_step(
        models[model_key], vqvae, c_tokens, c_actions,
        torch.tensor(f_actions, dtype=torch.long), horizon
    )

    # Decode context frames
    ctx_imgs = []
    for t in range(ctx):
        tok_grid = c_tokens[t].view(1, 8, 8).to(device)
        frame = vqvae.decode_from_tokens(tok_grid).clamp(0, 1).cpu()
        ctx_imgs.append(frame[0].permute(1, 2, 0).detach().numpy())

    total = ctx + horizon
    fig, axes = plt.subplots(3, total, figsize=(2*total, 6))

    for t in range(total):
        if t < ctx:
            axes[0, t].imshow(ctx_imgs[t])
            axes[1, t].imshow(ctx_imgs[t])
            axes[0, t].set_title(f"ctx {t+1}", fontsize=7, color="blue")
            axes[1, t].set_title(f"ctx {t+1}", fontsize=7, color="blue")
            axes[2, t].axis("off")
        else:
            p = t - ctx
            axes[0, t].imshow(gt_frames[p])
            axes[0, t].set_title(f"GT t+{p+1}", fontsize=7, color="green")
            axes[1, t].imshow(pred_frames[p])
            axes[1, t].set_title(f"pred t+{p+1}", fontsize=7, color="red")
            diff = np.abs(gt_frames[p] - pred_frames[p]).mean(axis=2)
            axes[2, t].imshow(diff, cmap="hot", vmin=0, vmax=0.3)
            axes[2, t].set_title(f"err", fontsize=7)
        for row in range(3):
            axes[row, t].axis("off")

    axes[0, 0].set_ylabel("Ground Truth", fontsize=10)
    axes[1, 0].set_ylabel("Predicted", fontsize=10)
    axes[2, 0].set_ylabel("Error", fontsize=10)

    plt.suptitle(f"{cfg['label']} — {ds_name}", fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f"ctx_{model_key}_{ds_name}_s{sample_idx}.png"),
                dpi=150, bbox_inches="tight")
    plt.show()

for mk in models:
    visualize_with_context("coinrun_test", model_key=mk, sample_idx=0, horizon=10)


### Action Conditioning Test

In [ ]:
def test_action_conditioning(ds_name, model_key="V1"):
    cfg = MODEL_CONFIGS[model_key]
    ctx = cfg["params"]["context_frames"]
    d = datasets_ctx5[ds_name] if ctx == 5 else datasets_ctx4[ds_name]
    model = models[model_key]

    ctx_tokens, ctx_actions, _, _, _ = d.get_sequence(0, horizon=1)

    predictions = []
    for action in range(min(5, 15)):
        modified_actions = ctx_actions.clone()
        modified_actions[-1] = action
        pred_frames, _ = predict_multi_step(
            model, vqvae, ctx_tokens, modified_actions,
            torch.tensor([], dtype=torch.long), horizon=1
        )
        predictions.append(pred_frames[0])

    n_actions = len(predictions)
    diffs = np.zeros((n_actions, n_actions))
    for i in range(n_actions):
        for j in range(n_actions):
            diffs[i, j] = np.mean((predictions[i] - predictions[j]) ** 2)

    fig, axes = plt.subplots(1, n_actions + 1, figsize=(n_actions*2 + 3, 3))
    fig.suptitle(f"Action Conditioning — {cfg['label']} on {ds_name}", fontsize=11)
    for i in range(n_actions):
        axes[i].imshow(predictions[i])
        axes[i].set_title(f"Act {i}", fontsize=8)
        axes[i].axis("off")
    im = axes[-1].imshow(diffs, cmap="viridis")
    axes[-1].set_title("Pairwise MSE", fontsize=8)
    plt.colorbar(im, ax=axes[-1], fraction=0.046)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f"action_cond_{model_key}_{ds_name}.png"), dpi=150)
    plt.show()

    avg_diff = diffs[np.triu_indices(n_actions, k=1)].mean()
    status = "✓ Using actions" if avg_diff > 0.001 else "✗ NOT using actions"
    print(f"  {cfg['label']} on {ds_name}: avg MSE={avg_diff:.6f} — {status}")

for mk in ["V1", "V3"]:
    test_action_conditioning("coinrun_train", model_key=mk)


### Save Results

In [ ]:
import json

# Build flat results for CSV
rows = []
for model_key in all_results:
    cfg = MODEL_CONFIGS[model_key]
    for ds_name in all_results[model_key]:
        game = ds_name.split("_")[0]
        split = ds_name.split("_")[1]
        for h in all_results[model_key][ds_name]:
            r = all_results[model_key][ds_name][h]
            rows.append({
                "model": cfg["label"], "model_key": model_key,
                "game": game, "split": split, "horizon": h,
                "psnr": r["psnr"], "ssim": r["ssim"], "token_acc": r["token_acc"],
            })

df = pd.DataFrame(rows)
df.to_csv(os.path.join(RESULTS_DIR, "all_model_results.csv"), index=False)
print("Results saved to CSV!")

# Also save as JSON
json_results = {}
for mk in all_results:
    json_results[mk] = {}
    for ds in all_results[mk]:
        json_results[mk][ds] = {str(h): {k: float(v) for k,v in m.items()}
                                  for h, m in all_results[mk][ds].items()}

with open(os.path.join(RESULTS_DIR, "all_model_results.json"), "w") as f:
    json.dump(json_results, f, indent=2)

print(f"\nAll outputs in {RESULTS_DIR}/")
for fname in sorted(os.listdir(RESULTS_DIR)):
    size = os.path.getsize(os.path.join(RESULTS_DIR, fname)) / 1024
    print(f"  {fname:50s} {size:.1f} KB")
